In [ ]:
import os, glob
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

device = "cuda" if torch.cuda.is_available() else "cpu"
class_names = ["meningioma", "glioma", "pituitary"]

saved = torch.load(glob.glob("/kaggle/input/**/slices_224.pt", recursive=True)[0])
X, y = saved["X"].float(), saved["y"]

csv_path = glob.glob("/kaggle/input/**/*slice_info*.csv", recursive=True)[0]
df = pd.read_csv(csv_path, dtype={"slice_id": str, "pid": str})
if "slice_ids" in saved:
    df = df.set_index("slice_id").loc[saved["slice_ids"]].reset_index()

assert len(df) == len(X) == len(y)
assert (df["label"].to_numpy() - 1 == y.numpy()).all(), "rows of X and the CSV don't line up"
idx = {s: np.where(df["split"] == s)[0] for s in ["train", "val", "test"]}
print(X.shape, {k: len(v) for k, v in idx.items()})

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class SliceDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        j = self.indices[i]
        return (X[j].repeat(3, 1, 1) - MEAN) / STD, y[j]

def predict(model, indices):
    dl = DataLoader(SliceDataset(indices), batch_size=64)
    model.eval()
    probs = []
    with torch.no_grad():
        for xb, _ in dl:
            probs.append(F.softmax(model(xb.to(device)), dim=1).cpu())
    return torch.cat(probs).numpy()

ckpt = glob.glob("/kaggle/input/**/resnet18_best.pth", recursive=True)[0]
baseline = models.resnet18()
baseline.fc = nn.Linear(baseline.fc.in_features, 3)
baseline.load_state_dict(torch.load(ckpt, map_location="cpu"))
baseline = baseline.to(device)

In [ ]:
val_probs = predict(baseline, idx["val"])
val_true = y[idx["val"]].numpy()
print("val accuracy:", round(float((val_probs.argmax(1) == val_true).mean()), 3))
print(classification_report(val_true, val_probs.argmax(1), target_names=class_names, digits=3))

In [ ]:
val_df = df.iloc[idx["val"]].copy()
val_df["true"], val_df["pred"] = val_true, val_probs.argmax(1)
errors = val_df[val_df.true != val_df.pred]
print(errors.groupby(["pid", "class_name"]).size().sort_values(ascending=False))

In [ ]:
def patient_vote(sub_df, probs):
    t = pd.DataFrame(probs, columns=[0, 1, 2])
    t["pid"], t["true"] = sub_df["pid"].values, sub_df["label"].values - 1
    g = t.groupby("pid").agg({0: "mean", 1: "mean", 2: "mean", "true": "first"})
    pred = g[[0, 1, 2]].values.argmax(1)
    return (pred == g["true"].values).mean(), len(g), g.assign(pred=pred)

def bootstrap_ci(sub_df, probs, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    correct = pd.Series((probs.argmax(1) == sub_df["label"].values - 1).astype(float))
    grp = correct.groupby(sub_df["pid"].values)
    sums, counts = grp.sum().values, grp.count().values
    accs = []
    for _ in range(n):
        pick = rng.integers(0, len(sums), len(sums))
        accs.append(sums[pick].sum() / counts[pick].sum())
    return np.percentile(accs, [2.5, 97.5])

pacc, npat, _ = patient_vote(val_df, val_probs)
print(f"val patient-level accuracy: {pacc:.3f} on {npat} patients")
print("95% interval for slice accuracy:", bootstrap_ci(val_df, val_probs).round(3))

In [ ]:
pid_to_check = "100820"
rows = val_df[val_df.pid == pid_to_check]
print(len(rows), "slices;", (rows.true != rows.pred).sum(), "wrong")

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax in axes.ravel():
    ax.axis("off")
for ax, (i, r) in zip(axes.ravel(), rows.head(10).iterrows()):
    ax.imshow(X[i, 0], cmap="gray")
    ax.set_title(f"true {class_names[r.true]}\npred {class_names[r.pred]}", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
pid_to_check = "MR017260F"
rows = val_df[val_df.pid == pid_to_check]
print(len(rows), "slices;", (rows.true != rows.pred).sum(), "wrong")

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax in axes.ravel():
    ax.axis("off")
for ax, (i, r) in zip(axes.ravel(), rows.head(10).iterrows()):
    ax.imshow(X[i, 0], cmap="gray")
    ax.set_title(f"true {class_names[r.true]}\npred {class_names[r.pred]}", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
for d in (df, val_df):
    d["id_format"] = np.where(d["pid"].str.startswith("MR"), "MR-prefixed", "numeric")
val_df["correct"] = val_df.true == val_df.pred

print(pd.crosstab(df["id_format"], df["class_name"]))
print(val_df.groupby("id_format").agg(patients=("pid", "nunique"),
                                      slices=("pid", "size"),
                                      accuracy=("correct", "mean")))

In [ ]:
test_df = df.iloc[idx["test"]].copy()
test_probs = predict(baseline, idx["test"])
test_pred = test_probs.argmax(1)
test_true = y[idx["test"]].numpy()

print(classification_report(test_true, test_pred, target_names=class_names, digits=3))
pacc, npat, _ = patient_vote(test_df, test_probs)
print(f"test patient-level accuracy: {pacc:.3f} on {npat} patients")
print("95% interval for slice accuracy:", bootstrap_ci(test_df, test_probs).round(3))

ConfusionMatrixDisplay(confusion_matrix(test_true, test_pred),
                       display_labels=class_names).plot(cmap="Blues")
plt.savefig("/kaggle/working/test_confusion.png", dpi=150)
plt.show()

test_df["true"], test_df["pred"] = test_true, test_pred
test_df.to_csv("/kaggle/working/test_predictions.csv", index=False)
print(test_df[test_df.true != test_df.pred].groupby(["pid", "class_name"]).size())

In [ ]:
val_df = df.iloc[idx["val"]].copy()
val_probs = predict(baseline, idx["val"])
val_df["true"], val_df["pred"] = y[idx["val"]].numpy(), val_probs.argmax(1)

test_df = df.iloc[idx["test"]].copy()
test_probs = predict(baseline, idx["test"])
test_df["true"], test_df["pred"] = y[idx["test"]].numpy(), test_probs.argmax(1)

for d in (df, val_df, test_df):
    d["id_format"] = np.where(d["pid"].str.startswith("MR"), "MR-prefixed", "numeric")

print((val_df.true != val_df.pred).sum(), "val errors;", (test_df.true != test_df.pred).sum(), "test errors")

In [ ]:
# 1) Glioma recall by ID group, on validation and test
for name, d in [("val", val_df), ("test", test_df)]:
    g = d[d.label == 2].copy()
    g["correct"] = g.true == g.pred
    print(name)
    print(g.groupby("id_format").agg(patients=("pid", "nunique"),
                                     slices=("pid", "size"),
                                     recall=("correct", "mean")))

# 2) How many numeric-ID gliomas does the model train on?
tr = df.iloc[idx["train"]]
print(tr[tr.label == 2].groupby("id_format").agg(patients=("pid", "nunique"),
                                                 slices=("pid", "size")))

# 3) Average image of each group
groups = {
    "glioma, MR ids":      (df.label == 2) & (df.id_format == "MR-prefixed"),
    "glioma, numeric ids": (df.label == 2) & (df.id_format == "numeric"),
    "meningioma":          df.label == 1,
    "pituitary":           df.label == 3,
}
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (name, mask) in zip(axes, groups.items()):
    ax.imshow(X[np.where(mask)[0], 0].mean(0), cmap="gray")
    ax.set_title(name, fontsize=9)
    ax.axis("off")
plt.show()

In [ ]:
rng = np.random.default_rng(1)
groups = {"glioma, MR ids":      (df.label == 2) & (df.id_format == "MR-prefixed"),
          "glioma, numeric ids": (df.label == 2) & (df.id_format == "numeric")}
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for row, (name, mask) in enumerate(groups.items()):
    for ax, i in zip(axes[row], rng.choice(np.where(mask)[0], 6, replace=False)):
        ax.imshow(X[i, 0], cmap="gray")
        ax.set_title(f"{name}\n{df['pid'].iloc[i]}", fontsize=8)
        ax.axis("off")
plt.tight_layout()
plt.savefig("/kaggle/working/glioma_id_groups.png", dpi=150)
plt.show()

In [ ]:
rows = []
for name, d, p in [("val", val_df, val_probs), ("test", test_df, test_probs)]:
    true, pred = d["true"].to_numpy(), p.argmax(1)
    pacc, n_pat, _ = patient_vote(d, p)
    lo, hi = bootstrap_ci(d, p)
    rep = classification_report(true, pred, target_names=class_names, output_dict=True)
    row = {"split": name, "slices": len(d), "patients": n_pat,
           "slice_accuracy": round(float((true == pred).mean()), 3),
           "slice_acc_ci_low": round(float(lo), 3), "slice_acc_ci_high": round(float(hi), 3),
           "patient_accuracy": round(float(pacc), 3)}
    for c in class_names:
        row[f"{c}_recall"] = round(rep[c]["recall"], 3)
        row[f"{c}_precision"] = round(rep[c]["precision"], 3)
        row[f"{c}_f1"] = round(rep[c]["f1-score"], 3)
    rows.append(row)
summary = pd.DataFrame(rows)
summary.to_csv("/kaggle/working/results_summary.csv", index=False)
print(summary.T)